In [1]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2 import service_account
import io

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'credentials.json'

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)

service = build('drive', 'v3', credentials=creds)

folder_id = '1BoaRum8pO-ITELHTcsrX5Fhsxmv3VeLX'

files = service.files().list(
    q=f"'{folder_id}' in parents and trashed=false",
    fields="files(id, name, mimeType)"
).execute()['files']

parquet_files = [f for f in files if f['name'].endswith('.parquet ')]
file_id = files[0]['id']  # or sort if needed

# download + read
request = service.files().get_media(fileId=file_id)
fh = io.BytesIO()

downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    _, done = downloader.next_chunk()

fh.seek(0)

0

In [12]:
import pandas as pd
df = pd.read_parquet(fh)

In [17]:
print(f'{df.shape=}')
df.isna().sum()

df.shape=(3724889, 20)


VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [18]:
df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]][:2]

,tpep_pickup_datetime,tpep_dropoff_datetime
0,2026-01-01 00:54:04,2026-01-01 00:59:37
1,2026-01-01 00:34:04,2026-01-01 00:39:47
